In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import glob
import re
import os
import warnings
warnings.filterwarnings("ignore")

def clean_column_name(name: str) -> str:
    """
    将变量的 long_name 转换为合法的 Parquet 列名：
    - 移除括号
    - 将空白字符替换为下划线
    - 去除首尾下划线
    """
    name = re.sub(r'[\(\)]', '', name)          # 移除括号
    name = re.sub(r'\s+', '_', name)            # 空白转下划线
    name = name.strip('_')
    return name

# 根据表定义的需要提取的所有变量名
REQUIRED_VARS = [
    'LON', 'LAT', 'GOT', 'GR', 'GF', 'GA', 'GFA', 'GEA', 'GEC', 'DQF',
    'OBIType', 'nominal_satellite_subpoint_lat', 'nominal_satellite_subpoint_lon',
    'nominal_satellite_height', 'geospatial_lat_lon_extent',
    'processing_parm_version_container', 'algorithm_product_version_container'
]

# 获取所有 NC 文件（请根据实际路径修改）
nc_files = glob.glob('D:/*.NC')   # 例如 './data/*.NC'
if not nc_files:
    raise FileNotFoundError("未找到任何 NC 文件，请检查路径。")

dataframes = []

for file in nc_files:
    # print(f"处理文件: {file}")
    ds = xr.open_dataset(file)

    # 检查所有必需变量是否存在于文件中
    missing = [v for v in REQUIRED_VARS if v not in ds.variables]
    if missing:
        raise ValueError(f"文件 {file} 缺少以下变量: {missing}")

    # 获取闪电组数的参考维度（LON 的维度）
    # 假设 LON 的维度名称为 'x'（从表4中 LON 的 shape 为 "x=" 推断）
    lon_var = ds['LON']
    # 确定 LON 的第一个维度（通常为 'x'）
    lon_dims = lon_var.dims
    if not lon_dims:
        raise ValueError("LON 变量没有维度，无法确定闪电组数量。")
    x_dim = lon_dims[0]          # 例如 'x'
    n_groups = ds.sizes[x_dim]    # 闪电组个数

    # 初始化当前文件的 DataFrame，行数为 n_groups
    df = pd.DataFrame(index=range(n_groups))

    # 遍历每个变量
    for var_name in REQUIRED_VARS:
        var = ds[var_name]
        dims = var.dims
        shape = var.shape

        # 获取 long_name，若无则用变量名
        long_name = var.attrs.get('long_name', var_name)
        col_name = clean_column_name(long_name)

        # 判断该变量是否为数组变量（即与 LON 共享相同的闪电组维度）
        if dims and dims[0] == x_dim and shape[0] == n_groups:
            # 数组变量：直接取数据，长度应与 n_groups 一致
            data = var.values
            # 处理掩码数组
            if hasattr(data, 'mask'):
                data = data.filled(np.nan)
            df[col_name] = data
        else:
            # 标量变量：获取其值（可能是数组、标量或列表）
            # 使用 item() 将 0 维数组转换为 Python 标量，否则保留为数组/列表
            val = var.values
            if val.ndim == 0:
                val = val.item()
            # 将该值复制到所有行
            df[col_name] = [val] * n_groups

    # 可选：添加源文件名便于追溯
    df['source_file'] = os.path.basename(file)

    dataframes.append(df)
    ds.close()

# 合并所有文件的数据
if dataframes:
    combined_df = pd.concat(dataframes, ignore_index=True)
    # 保存为 Parquet 文件
    combined_df.to_parquet('all_lmig_data_jun.parquet', index=False, compression='snappy')
    print(f"成功合并 {len(dataframes)} 个文件，共 {len(combined_df)} 行数据，已保存至 all_lmig_data.parquet")
else:
    print("未提取到任何数据。")

In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import glob
import re
import os
import warnings
from tqdm import tqdm  # 只添加这一行导入

# 忽略所有警告（或根据需要选择忽略特定类别）
warnings.filterwarnings("ignore")   # 可选：更精细地忽略 FutureWarning 和 SerializationWarning

def clean_column_name(name: str) -> str:
    """
    将变量的 long_name 转换为合法的 Parquet 列名：
    - 移除括号
    - 将空白字符替换为下划线
    - 去除首尾下划线
    """
    name = re.sub(r'[\(\)]', '', name)          # 移除括号
    name = re.sub(r'\s+', '_', name)            # 空白转下划线
    name = name.strip('_')
    return name

# 根据表4定义的需要提取的所有变量名
REQUIRED_VARS = [
    'LON', 'LAT', 'GOT', 'GR', 'GF', 'GA', 'GFA', 'GEA', 'GEC', 'DQF',
    'OBIType', 'nominal_satellite_subpoint_lat', 'nominal_satellite_subpoint_lon',
    'nominal_satellite_height', 'geospatial_lat_lon_extent',
    'processing_parm_version_container', 'algorithm_product_version_container'
]

# 获取所有 NC 文件（请根据实际路径修改）
nc_files = glob.glob('D:/Database/Work Program/PyProject/Weather Finance Research/Data_2306/*.NC')   # 例如 './data/*.NC'
if not nc_files:
    raise FileNotFoundError("未找到任何 NC 文件，请检查路径。")

dataframes = []

# 添加 tqdm 进度条包装文件列表
for file in tqdm(nc_files, desc="处理NC文件", unit="file"):
    # print(f"处理文件: {file}")
    ds = xr.open_dataset(file)

    # 检查所有必需变量是否存在于文件中
    missing = [v for v in REQUIRED_VARS if v not in ds.variables]
    if missing:
        raise ValueError(f"文件 {file} 缺少以下变量: {missing}")

    # 获取闪电组数的参考维度（LON 的维度）
    # 假设 LON 的维度名称为 'x'（从表4中 LON 的 shape 为 "x=" 推断）
    lon_var = ds['LON']
    # 确定 LON 的第一个维度（通常为 'x'）
    lon_dims = lon_var.dims
    if not lon_dims:
        raise ValueError("LON 变量没有维度，无法确定闪电组数量。")
    x_dim = lon_dims[0]          # 例如 'x'
    n_groups = ds.sizes[x_dim]    # 闪电组个数

    # 初始化当前文件的 DataFrame，行数为 n_groups
    df = pd.DataFrame(index=range(n_groups))

    # 遍历每个变量
    for var_name in REQUIRED_VARS:
        var = ds[var_name]
        dims = var.dims
        shape = var.shape

        # 获取 long_name，若无则用变量名
        long_name = var.attrs.get('long_name', var_name)
        col_name = clean_column_name(long_name)

        # 判断该变量是否为数组变量（即与 LON 共享相同的闪电组维度）
        if dims and dims[0] == x_dim and shape[0] == n_groups:
            # 数组变量：直接取数据，长度应与 n_groups 一致
            data = var.values
            # 处理掩码数组
            if hasattr(data, 'mask'):
                data = data.filled(np.nan)
            df[col_name] = data
        else:
            # 标量变量：获取其值（可能是数组、标量或列表）
            # 使用 item() 将 0 维数组转换为 Python 标量，否则保留为数组/列表
            val = var.values
            if val.ndim == 0:
                val = val.item()
            # 将该值复制到所有行
            df[col_name] = [val] * n_groups

    # 可选：添加源文件名便于追溯
    df['source_file'] = os.path.basename(file)

    dataframes.append(df)
    ds.close()

# 合并所有文件的数据
if dataframes:
    combined_df = pd.concat(dataframes, ignore_index=True)
    # 保存为 Parquet 文件
    combined_df.to_parquet('all_lmig_data_jun.parquet', index=False, compression='snappy')
    print(f"成功合并 {len(dataframes)} 个文件，共 {len(combined_df)} 行数据，已保存至 all_lmig_data.parquet")
else:
    print("未提取到任何数据。")

处理NC文件: 100%|██████████| 42677/42677 [08:36<00:00, 82.70file/s]


成功合并 42677 个文件，共 827592 行数据，已保存至 all_lmig_data.parquet
